# Import Data

In [1]:
import pandas as pd
import dask.dataframe as dd
import numpy as np
import gc
from tqdm import tqdm

In [2]:
df_credit_clean = pd.read_parquet("data/base/df_credit_clean_dec24_may25.parquet")
df_time_diff = dd.read_parquet("data/feature_engineering/credit/v3/df_time_diff.parquet")
df_freq = dd.read_parquet("data/feature_engineering/credit/v3/df_freq.parquet")
df_monetary = dd.read_parquet("data/feature_engineering/credit/v3/df_monetary.parquet")
df_unique_count = dd.read_parquet("data/feature_engineering/credit/v3/df_unique_cnt.parquet")

In [3]:
def compute_grouped_rolling_avg_trnx_hour(
    df: pd.DataFrame,
    group_col: str,
    datetime_col: str,
    windows: list[str]
) -> pd.DataFrame:
    """
    Compute grouped rolling average of transaction hour per group
    over time windows
    """
    df = df.copy()
    df[datetime_col] = pd.to_datetime(df[datetime_col])
    df['TrnxHour'] = df[datetime_col].dt.hour

    # container for results
    result = []

    # apply group-wise rolling
    for group_value, group_df in tqdm(
        df.groupby(group_col), desc="Processing TrnxHour Rolling Avg"):
        group_df = group_df.sort_values(datetime_col).set_index(datetime_col)
        
        # for each window, apply group-wise rolling
        for window in windows:
            col_name = f'AvgTrnxHourL{window}'
            group_df[col_name] = group_df['TrnxHour'].rolling(
                window=window, min_periods=1
            ).mean()

        # restore group column
        group_df[group_col] = group_value
        result.append(group_df.reset_index())

    # concatenate all groupes
    df_final = pd.concat(result, ignore_index=False)
    return df_final

In [4]:
windows=['15min','1h','1d','14d','30d']

df_trnx_hour = compute_grouped_rolling_avg_trnx_hour(
    df=df_credit_clean,
    group_col='PANNumber',
    datetime_col='Transaction Datetime',
    windows=windows
)

Processing TrnxHour Rolling Avg: 100%|██████████| 38802/38802 [02:28<00:00, 261.18it/s]


In [5]:
df_trnx_hour.to_parquet("data/feature_engineering/credit/v3/df_trnx_hour.parquet")

del df_credit_clean
del df_trnx_hour

In [3]:
df_credit_clean = dd.read_parquet("data/base/df_credit_clean_dec24_may25.parquet")
df_trnx_hour = dd.read_parquet("data/feature_engineering/credit/v3/df_trnx_hour.parquet")

In [4]:
channel_cols = [
    'PANNumber'
	, 'Transaction Serial No'
    , 'Transaction Datetime'
    , 'Product Indicator'
    , 'Transaction Amount'
    , 'MCC'
    , 'MCC Details'
    , 'MCC Trnx Category Code'
    , 'MCC Category'
    , 'Country Code'
    , 'Card Acceptor Terminal ID'
    , 'Card Acceptor ID'
    , 'Card Acceptor Name'
    , 'Card Acceptor City'
    , 'Card Acceptor Region Code'
    , 'Card Acceptor Country Code'
    , 'Cat Card Acceptor Name'
    , 'Currency Code'
    , 'Confirmed'
]

# Join Data

In [5]:
from src.credit_card_config import (
    time_shift_config,
    time_windows,
    freq_config,
    monetary_config_1,
    monetary_config_2,
    monetary_config_3,
    monetary_config_4,
    monetary_config_5,
    monetary_config_6,
    unique_count_config
)
import warnings
warnings.filterwarnings('ignore')

all_monetary_configs = (
    monetary_config_1 + monetary_config_2 + monetary_config_3 +
    monetary_config_4 + monetary_config_5 + monetary_config_6
)

In [6]:
# Define the common keys for merging
merge_keys = ["Transaction Serial No"]

# Helper function to extract columns based on config and time windows
def extract_columns(config, time_windows):
    return [cfg["windows"][win] for cfg in config for win in time_windows]

# Extract columns
time_shift_cols = list(time_shift_config.keys())
freq_cols = extract_columns(freq_config, time_windows)
unique_count_cols = extract_columns(unique_count_config, time_windows)
monetary_cols = extract_columns(all_monetary_configs, time_windows)
monetary_cols = [x for x in monetary_cols if x not in ['Sum_Amt_L15M', 'Sum_Amt_L1H', 'Sum_Amt_L1D', 'Sum_Amt_L7D']]

windows=['15min','1h','1d','14d','30d']
trnx_hour_cols = [col for col in df_trnx_hour.columns if any(w in col for w in windows)]

In [7]:
from functools import reduce
import gc

# Subset dataframes
df_time_diff = df_time_diff[merge_keys + time_shift_cols]
df_freq = df_freq[merge_keys + freq_cols]
df_monetary = df_monetary[merge_keys + monetary_cols]
df_unique_count = df_unique_count[merge_keys + unique_count_cols]
df_trnx_hour = df_trnx_hour[merge_keys + trnx_hour_cols]

# Optional but might help: use repartition on merge keys (improtve merge performance on large datasets)
# use npartitions depending on your data size and RAM
df_time_diff = df_time_diff.repartition(npartitions=10)
df_freq = df_freq.repartition(npartitions=10)
df_monetary = df_monetary.repartition(npartitions=10)
df_unique_count = df_unique_count.repartition(npartitions=10)
df_trnx_hour = df_trnx_hour.repartition(npartitions=10)

# List all feature dataframes
dfs_to_merge = [df_time_diff, df_freq, df_monetary, df_unique_count, df_trnx_hour]

# Perform Dask merge
df_credit_final = reduce(
    lambda left, right: dd.merge(left, right, on=merge_keys, how="outer"),
    dfs_to_merge
)

del df_time_diff
del df_freq
del df_monetary
del df_unique_count
del df_trnx_hour
gc.collect()

# Additional TSCF Features
tscf_cols = [col for col in df_credit_clean.columns if col not in channel_cols]
selected_channel_cols = [
    # 'PANNumber'
	'Transaction Serial No'
    , 'Transaction Datetime'
    , 'Transaction Amount'
    , 'MCC Category'
    , 'Country Code'
    , 'Cat Card Acceptor Name'
    , 'Currency Code'
    , 'Confirmed'
]

# Merge with additional features
df_credit_final = df_credit_final.merge(
    df_credit_clean[selected_channel_cols + tscf_cols],
    on=merge_keys,
    how="left",
)

# Optional: persist final result in memory
df_credit_final = df_credit_final.persist()

In [16]:
# df_credit_final = df_credit_final.drop(['PANNumber'], axis=1)
# df_credit_final = df_credit_final.drop(['PANNumber_x'], axis=1)

In [8]:
# df_credit_final['Confirmed'] = df_credit_final['Confirmed'].fillna(False)
# df_credit_final.to_parquet(
#     "data/feature_engineering/credit/v3/df_credit_final.parquet",
# )
df_credit_final['Confirmed'] = df_credit_final['Confirmed'].fillna(False)
df_credit_final.to_parquet(
    "data/feature_engineering/credit/v3/df_credit_final_20250620.parquet",
)